# Trabalho de EST2 - G13

Fazer as regressões Ridge, Lasso e ElasticNet com a variável dependente “lwage” (salário-hora da esposa em logaritmo neperiano) e todas as demais variáveis da base de dados são variáveis explicativas (todas essas variáveis tentam explicar o salário-hora da esposa). No pdf você deve colocar a rotina utilizada, mostrar em uma tabela as estatísticas dos modelos (RMSE e R2) e concluir qual o melhor modelo entre os três, e mostrar o resultado da predição com intervalos de confiança para os seguintes valores:  
  
husage = 40 (anos – idade do marido)  
husunion = 0 (marido não possui união estável)  
husearns = 600 (US renda do marido por semana)  
huseduc = 13 (anos de estudo do marido)  
husblck = 1 (o marido é preto)  
hushisp = 0 (o marido não é hispânico)  
hushrs = 40 (horas semanais de trabalho do marido)  
kidge6 = 1 (possui filhos maiores de 6 anos)  
earns = 355.5 (US renda da esposa por semana)   
age = 38 (anos – idade da esposa)  
black = 0 (a esposa não é preta)  
educ = 13 (anos de estudo da esposa)  
hispanic = 1 (a esposa é hispânica)  
union = 0 (esposa não possui união estável)  
exper = 18 (anos de experiência de trabalho da esposa)  
kidlt6 = 1 (possui filhos menores de 6 anos)  

obs: lembre-se de que a variável dependente “lwage” já está em logarítmo, portanto voçê não precisa aplicar o logaritmo nela para fazer as regressões, mas é necessário aplicar o antilog para obter o resultado da predição.

Obs2: o valor faltante de 'earns' deverá ser preenchido com a média dessa variável

In [52]:
# Bibliotecas a serem utilizadas
import numpy as np
import pandas as pd
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import  root_mean_squared_error

# Para medir o tempo do processo
import time

In [53]:
# Vamos utilizar a base de dados "trabalhosalarios" 
# Carregando o dataset
df = pd.read_csv('trabalhosalarios.csv')

In [54]:
df.head()

,husage,husunion,husearns,huseduc,husblck,hushisp,hushrs,kidge6,earns,age,black,educ,hispanic,union,exper,kidlt6,lwage
0,56,0,1500,14,0,0,40,1,100.0,49,0,12,0,0,31,0,1.897120
1,31,0,800,17,0,0,40,0,480.0,29,0,14,0,0,9,0,2.484907
2,33,0,950,13,0,0,60,0,455.0,30,0,12,0,0,12,1,2.431418
3,34,0,1000,14,0,0,50,1,102.0,31,0,12,0,0,13,0,1.629241
4,42,0,730,14,0,0,40,1,300.0,41,0,12,0,0,23,0,2.302585


In [55]:
y = df[['lwage']].to_numpy() 
y = y.ravel() # garantir o shape devido a regressão Lasso aguardar y com o shape (n,) ao inves de (n,1)
X = df.drop(['lwage'], axis=1)

In [56]:
# Vamos particionar o dataset em 80% para treinamento e 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [57]:
# Vamos checar as dimensoes das bases de treinamento e teste
print(f"Total de linhas e colunas do dataset : {df.shape} ")
print(f"Total de linhas e colunas do Treino : {X_treino.shape}")
print(f"Total de linhas e colunas do Teste : {X_teste.shape}")

Total de linhas e colunas do dataset : (2574, 17) 
Total de linhas e colunas do Treino : (2059, 16)
Total de linhas e colunas do Teste : (515, 16)


In [58]:
# A base de treinamento possui 2059 linhas (observações) e  16 colunas (variaveis)
# A base de teste possui 515 linhas (observações) e 16 colunas (variaveis)

In [59]:
print(df.columns)

Index(['husage', 'husunion', 'husearns', 'huseduc', 'husblck', 'hushisp',
       'hushrs', 'kidge6', 'earns', 'age', 'black', 'educ', 'hispanic',
       'union', 'exper', 'kidlt6', 'lwage'],
      dtype='str')


In [60]:
# Vamos listar as colunas que necessitam padronizar (contínuas) e as que não devem ser padronizadas (binárias)
col_continuas = ['husage', 'husearns', 'huseduc', 'hushrs', 'earns', 'age', 'educ', 'exper']
col_binarias = ['husunion','husblck', 'hushisp', 'kidge6', 'black', 'hispanic', 'union', 'kidlt6']

In [61]:
#############################################################
#                     Regressões                            #
#############################################################

In [62]:
alphas = np.logspace(-3, 2, 100)

models = {
    "Ridge": RidgeCV(alphas=alphas, cv=5),
    "Lasso": LassoCV(alphas=alphas, cv=5, random_state = 42),
    "ElasticNet": ElasticNetCV(alphas=alphas, cv=5, random_state = 42)
    
}

In [ ]:
trained_models = {} # Dicionario para guardar os treinos

# Dataframe para guardar os coeficientes
df_coefs = pd.DataFrame({"Variavel": X_treino.columns}).set_index("Variavel")

# Cria um preprocessamento para transformar apenas as variáveis contínuas e passar as binárias
processamento_X = ColumnTransformer(
    transformers=[
        ("cont", StandardScaler(), col_continuas),
        ("bin", "passthrough", col_binarias)
    ]
)

# loop para iterar entre os modelos
for name, model_reg in models.items():
    start_time = time.time() # inicia contagem de tempo

    model = Pipeline(steps=[
        ("preprocessamento", processamento_X), # preprocessa as variaveis independente
        ("modelo", model_reg) # aplica um modelo
    ])
    
    # Começa aqui, depois passa pelo pipeline para tratar o X
    
    model.fit(X_treino, y_treino)  # Treina

    y_hat = model.predict(X_teste) # faz a predição
    
    train_time = round((time.time() - start_time), 2) # calcula o tempo do processo
    
    rmse_calc = round(root_mean_squared_error(y_teste, y_hat),2) # calcula o RMSE
    
    r2_treino = round(model.score(X_treino, y_treino) * 100, 2) # calcula o R² do Treino
    
    r2_teste = round(model.score(X_teste, y_teste) * 100,2) # calcula o R² do Teste

    trained_models[name] = {
        "model": model,
        "alpha": model.named_steps['modelo'].alpha_,
        "RMSE" : rmse_calc,
        "R2_Treino" : r2_treino,
        "R2_Teste" : r2_teste,
        "tempo": train_time
    } # salva no dicionario
    
    # Salva os coefientes no Dataframe
    df_coefs[name] = pd.Series(model.named_steps['modelo'].coef_, index= X_treino.columns)

    print(f'Tempo total de treinamento e predição {name} : {train_time} segundos')
    
    # Guarda em um Dataframe
    df_trained_models = pd.DataFrame(trained_models).T

Tempo total de treinamento e predição Ridge : 0.8 segundos
Tempo total de treinamento e predição Lasso : 0.05 segundos
Tempo total de treinamento e predição ElasticNet : 0.05 segundos


In [64]:
# Exibe as medidas
print(df_trained_models[['RMSE', 'R2_Treino', 'R2_Teste', 'tempo', 'alpha']])

            RMSE R2_Treino R2_Teste tempo      alpha
Ridge       0.26      68.2    72.96   0.8  31.257158
Lasso       0.26     68.08    72.86  0.05   0.002535
ElasticNet  0.26      68.1    72.87  0.05   0.004535


Com base nos valores de R², o modelo Ridge apresentou um desempenho ligeiramente melhor em relação aos demais modelos. O fato de apresentar um Alfa alto indica uma forte regularização e que existem dados que apresentam forte colinearidade.

In [65]:
# Exibe os coeficientes de cada modelo
print(df_coefs)

             Ridge     Lasso  ElasticNet
Variavel                                
husage    0.008471  0.006813    0.006897
husunion  0.040691  0.038584    0.038895
husearns  0.011304  0.011098    0.011177
huseduc  -0.012234 -0.008918   -0.009260
husblck   0.380450  0.385814    0.384898
hushisp   0.008956  0.000156    0.000541
hushrs    0.055329  0.056619    0.056707
kidge6   -0.006283  0.000000    0.000000
earns     0.011215  0.001480    0.002532
age       0.016858 -0.000000   -0.000000
black    -0.029532 -0.003479   -0.008143
educ      0.006930  0.000000    0.000000
hispanic -0.039040 -0.000000   -0.000000
union    -0.021333 -0.000000   -0.000188
exper     0.049857  0.034935    0.036583
kidlt6    0.063433  0.044982    0.046487


In [66]:
# obter o valor médio de 'earns'
earns_medio = X_treino['earns'].mean()

In [69]:
earns_medio

np.float64(373.24477901894124)

In [67]:
# Vamos criar uma nova observação para prever o valor da hora salario da esposa

# husage = 40 (anos – idade do marido)  
# husunion = 0 (marido não possui união estável)  
# husearns = 600 (US$ renda do marido por semana)  
# huseduc = 13 (anos de estudo do marido)  
# husblck = 1 (o marido é preto)  
# hushisp = 0 (o marido não é hispânico)  
# hushrs = 40 (horas semanais de trabalho do marido)  
# kidge6 = 1 (possui filhos maiores de 6 anos)
# earns = valor médio do treino  
# age = 38 (anos – idade da esposa)  
# black = 0 (a esposa não é preta)  
# educ = 13 (anos de estudo da esposa)  
# hispanic = 1 (a esposa é hispânica)  
# union = 0 (esposa não possui união estável)  
# exper = 18 (anos de experiência de trabalho da esposa)  
# kidlt6 = 1 (possui filhos menores de 6 anos) 
X_novo = pd.DataFrame({
    'husage': [40],
    'husunion': [0],
    'husearns' : [600],
    'huseduc': [13],
    'husblck': [1],
    'hushisp': [0],
    'hushrs': [40],
    'kidge6': [1],
    'earns': [earns_medio],
    'age': [38],
    'black': [0],
    'educ': [13],
    'hispanic': [1],
    'union': [0],
    'exper': [18],
    'kidlt6': [1]  
})

In [68]:
# calcular a predição para o novo valor

# cria um dataframe para guardar as medidas
df_pred = pd.DataFrame(columns=['y_novo', 'IC_inf', 'IC_sup'])

# insere no dataframe as predições calculadas de cada modelo convertida para o anti-log e inclui
# os intervalos de confiança para cada modelo

n = len(y_treino) # tamanho da amostra de treino
s = y_treino.std() # desvio padrao da amostra de treino
dam = s/np.sqrt(n) # distribuicao da amostragem da media

for name, info in trained_models.items():
    # calcula a predição
    y_hat_log = info['model'].predict(X_novo)[0]
    
    # calcula os Intervalos de confiança
    IC_inf = y_hat_log + (norm.ppf(0.025) * dam)
    IC_sup = y_hat_log - (norm.ppf(0.025) * dam)
    
    # converte e armazena
    df_pred.loc[name] = [round(np.exp(y_hat_log),2), round(np.exp(IC_inf), 2), round(np.exp(IC_sup), 2)]

print(df_pred)

            y_novo  IC_inf  IC_sup
Ridge         9.30    9.09    9.52
Lasso         9.17    8.96    9.38
ElasticNet    9.18    8.97    9.39
